# 限制循环步数，但被动退出

In [1]:
from typing import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError
from langgraph.graph import StateGraph, START
from loguru import logger


# 1、声明状态(空状态）
class EmptyState(TypedDict):
    pass


# 2、声明循环节点
def loop_node(state: EmptyState, config: RunnableConfig) -> EmptyState:
    cur_step = config["metadata"]["langgraph_step"]
    logger.info("loop_node,cur_step:{}", cur_step)


# 3、构建图
builder = StateGraph(state_schema=EmptyState)
builder.add_node("loop_node", loop_node)
builder.add_edge(START, "loop_node")
# 循环指向
builder.add_edge("loop_node", "loop_node")

graph = builder.compile()
try:
    graph.invoke({}, config={"recursion_limit": 10})
except GraphRecursionError as e:
    logger.info("超步数量达到最大限制,抛出异常:{}", e)

2026-07-22 16:56:44.339 | INFO     | __main__:loop_node:17 - loop_node,cur_step:1
2026-07-22 16:56:44.340 | INFO     | __main__:loop_node:17 - loop_node,cur_step:2
2026-07-22 16:56:44.340 | INFO     | __main__:loop_node:17 - loop_node,cur_step:3
2026-07-22 16:56:44.341 | INFO     | __main__:loop_node:17 - loop_node,cur_step:4
2026-07-22 16:56:44.341 | INFO     | __main__:loop_node:17 - loop_node,cur_step:5
2026-07-22 16:56:44.342 | INFO     | __main__:loop_node:17 - loop_node,cur_step:6
2026-07-22 16:56:44.342 | INFO     | __main__:loop_node:17 - loop_node,cur_step:7
2026-07-22 16:56:44.343 | INFO     | __main__:loop_node:17 - loop_node,cur_step:8
2026-07-22 16:56:44.343 | INFO     | __main__:loop_node:17 - loop_node,cur_step:9
2026-07-22 16:56:44.343 | INFO     | __main__:loop_node:17 - loop_node,cur_step:10
2026-07-22 16:56:44.344 | INFO     | __main__:<module>:31 - 超步数量达到最大限制,抛出异常:Recursion limit of 10 reached without hitting a stop condition. You can increase the limit by setting t